![Egeria Logo](https://raw.githubusercontent.com/odpi/egeria/main/assets/img/ODPi_Egeria_Logo_color.png)

### Coco Pharmaceuticals Labs

----

# Creating the system data layer

The systems inventory tells [Gary Geeke](https://egeria-project.org/practices/coco-pharmaceuticals/personas/gary-geeke/) *which* systems Coco Pharmaceuticals runs, and the mapping links them to the solution components of the [strategic information supply chains](https://egeria-project.org/practices/coco-pharmaceuticals/scenarios/defining-information-supply-chains/overview/).  Neither says what data the systems hold.  The strategic digital products in the next step are filled from that data, so before they are published Gary describes the data layer of every system that supplies or receives them.

Each of these systems keeps its data in its own PostgreSQL schema, and each estate has its own database:

| Estate | Database | Systems |
|---|---|---|
| **Coco core** | `coco_pharma` | 22 |
| **Austin** | `austin_systems` | 27 |
| **Bucharest** | `bucharest_systems` | 17 |

For each system this workbook builds the same small structure:

* a **`DatabaseManager`** software capability - the system's database layer - linked to the system's `SoftwareServer` with the **`SupportedSoftwareCapability`** relationship;
* a **`DeployedDatabaseSchema`** asset for the system's schema, linked to the database manager with the **`CapabilityAssetUse`** relationship.  The asset is created from the *PostgreSQL Relational Database Schema* [template](https://egeria-project.org/concepts/template/), so it comes with the `Connection`, `ConnectorType` and `Endpoint` that an integration connector needs to reach the schema;
* a **`CatalogTarget`** relationship from the `JDBCDatabaseCataloguer` integration connector to the schema asset.  The cataloguer then reads the schema's tables and columns from the database, catalogues them under the schema asset, and keeps them up to date as the systems change.

The work is shared between two people.  Gary creates the database managers, because the systems sit in governance zones that only the infrastructure team may update, and the `SupportedSoftwareCapability` relationship changes the system.  [Peter Profile](https://egeria-project.org/practices/coco-pharmaceuticals/personas/peter-profile/) then takes over the data side: creating the schema assets, linking them to the database managers, and setting up their cataloguing.

----

In [ ]:
# Show the structure built for each system

from pyegeria import load_mermaid, render_mermaid
load_mermaid()

render_mermaid("""
flowchart LR
    S["SoftwareServer<br/>the system"] -- SupportedSoftwareCapability --> D["DatabaseManager<br/>its database layer"]
    D -- CapabilityAssetUse --> A["DeployedDatabaseSchema<br/>+ Connection, Endpoint"]
    J["JDBCDatabaseCataloguer<br/>integration connector"] -- CatalogTarget --> A
    A -. catalogued by the connector .-> T["RelationalTable<br/>RelationalColumn"]
""")

----

## The systems and their schemas

The list of systems is in **`data/system-databases.csv`** - one row per system schema, taken from the scripts that create the system databases (`compose-configs/egeria-quickstart/docker-entrypoint-initdb.d/data/coco_systems/<database>/<schema>.sql`).  Each row gives the estate, the database and schema, the qualified name of the system in Egeria, the schema's description, and the number of tables the schema defines, which is used to check the cataloguing at the end.

----

In [ ]:
# Load the list of system schemas

import csv
from pathlib import Path
from collections import Counter, defaultdict

DATA_DIR = Path('./data').resolve()

with open(DATA_DIR / 'system-databases.csv', newline='', encoding='utf-8') as fh:
    system_databases = list(csv.DictReader(fh))

ESTATES = ["Coco core", "Austin", "Bucharest"]
systems_by_estate = Counter(row['estate'] for row in system_databases)
tables_by_estate  = Counter()
for row in system_databases:
    tables_by_estate[row['estate']] += int(row['table_count'])

print(f"{len(system_databases)} system schemas defining {sum(tables_by_estate.values())} tables")
for estate in ESTATES:
    print(f"  {estate:10} {systems_by_estate[estate]:3} schemas  {tables_by_estate[estate]:4} tables")

print()
for key, value in system_databases[0].items():
    print(f"  {key}: {value}")

In [ ]:
# Initialize pyegeria

import os
view_server = os.environ.get("EGERIA_VIEW_SERVER","qs-view-server")
url = os.environ.get("EGERIA_VIEW_SERVER_URL","https://host.docker.internal:9443")
user_pwd = os.environ.get("EGERIA_USER_PASSWORD")
egeria_width = 150

from IPython.display import display, Markdown

import json

# The classification explorer client is used to look up elements and their relationships.
# The asset maker client creates the database managers, the schema assets and the relationships between them.
# The runtime manager client is for controlling Egeria's servers.
from pyegeria import ClassificationExplorer, AssetMaker, RuntimeManager

def connect_as(new_user_id):
    """Create the clients for new_user_id.  The rest of the workbook uses whichever user connected last."""
    global user_id, classification_explorer, asset_maker, runtime_manager
    user_id = new_user_id
    classification_explorer = ClassificationExplorer(view_server, url, user_id, user_pwd)
    asset_maker = AssetMaker(view_server, url, user_id, user_pwd)
    runtime_manager = RuntimeManager(view_server, url, user_id, user_pwd, timeout=300)
    refresh_tokens()
    print("\nAccessing view server " + view_server + " on platform " + url + " for user " + user_id + "\n")

def refresh_tokens():
    """Create a new bearer token and share it between the clients."""
    token = asset_maker.create_egeria_bearer_token()
    classification_explorer.set_bearer_token(token)
    runtime_manager.set_bearer_token(token)

connect_as("garygeeke")   # Gary can update the systems' zones; the SupportedSoftwareCapability relationship changes the system

----

## Checking the starting point

The schema assets are created from the template that the PostgreSQL content pack supplies, and catalogued by the `JDBCDatabaseCataloguer` connector that runs in the `qs-integration-daemon`.  Both have fixed unique identifiers.  The template's placeholders are filled from the values below: the location of the PostgreSQL server that hosts the three system databases, and the secrets collection holding the credentials the cataloguer uses to connect to it - the same server and secrets that `setting-up-the-data-hub.ipynb` uses for `coco_data_hub`.

Every system's qualified name is then resolved to a GUID.  A system that does not resolve is reported and skipped: a missing Coco core system means `CocoComboArchive.omarchive` is not loaded, and a missing Austin or Bucharest system means [extending-the-systems-inventory.ipynb](../extending-the-systems-inventory/extending-the-systems-inventory.ipynb) has not been run.

----

In [ ]:
# The template, the cataloguer and the PostgreSQL server

refresh_tokens()

POSTGRES_SCHEMA_TEMPLATE_GUID = "82a5417c-d882-4271-8444-4c6a996a8bfc"   # PostgreSQL Relational Database Schema template
JDBC_CATALOGUER_GUID          = "70dcd0b7-9f06-48ad-ad44-ae4d7a7762aa"   # JDBCDatabaseCataloguer integration connector
JDBC_CATALOGUER_NAME          = "JDBCDatabaseCataloguer"

postgres_host_identifier = "host.docker.internal"
postgres_port_number     = "5442"
postgres_server_name     = "Coco PostgreSQL Server 1"
secrets_store_path_name  = "secrets/integration.omsecrets"
secrets_collection_name  = "PostgreSQL Server Secret"

DATABASE_MANAGER_TYPE = "PostgreSQL database manager (RDBMS)"            # deployedImplementationType of the capability

def schema_qualified_name(row):
    """The qualified name the template gives the schema asset."""
    return f"PostgreSQL Relational Database Schema::{postgres_server_name}::{row['database_name']}.{row['schema_name']}"

def database_manager_qualified_name(row):
    return f"DatabaseManager::{row['system_qualified_name']}"

for guid, name in [(POSTGRES_SCHEMA_TEMPLATE_GUID, "schema template"), (JDBC_CATALOGUER_GUID, JDBC_CATALOGUER_NAME)]:
    element = classification_explorer.get_element_by_guid(guid, graph_query_depth=0)
    if isinstance(element, dict):
        print(f"Found the {name}: {element.get('properties', {}).get('qualifiedName')}")
    else:
        print(f"The {name} ({guid}) is not in Egeria - is the PostgreSQL content pack loaded?")

# Retrieve the unique id (GUID) for the integration daemon.  This is used to force the cataloguing to happen immediately
# and to report on the cataloguer.  The platform may have catalogued the daemon more than once, under different platform
# URLs (for example https://localhost:9443 as well as https://host.docker.internal:9443), so prefer the element whose
# qualified name uses the URL this workbook connects to.
integration_daemon_guid = None
servers = runtime_manager.get_servers_by_name(filter_string="qs-integration-daemon")
if servers and not isinstance(servers, str):
    for server in servers:
        if not server:
            continue
        server_guid = server["elementHeader"]["guid"]
        server_qn   = server.get("properties", {}).get("qualifiedName", "")
        print(f"Found {server_qn}: {server_guid}")
        if integration_daemon_guid is None or url in server_qn:
            integration_daemon_guid = server_guid
print(f"Integration daemon: {integration_daemon_guid}")

In [ ]:
# Resolve every system to a GUID

refresh_tokens()

def guid_of(qualified_name):
    """Return the GUID of the element with this qualified name, or None."""
    result = classification_explorer.get_element_guid_by_unique_name(qualified_name)
    return result if isinstance(result, str) and result != "No elements found" else None

def collect_guids(obj, found=None):
    """Walk a pyegeria response and gather every 'guid' value in it."""
    if found is None: found = set()
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == 'guid' and isinstance(v, str): found.add(v)
            else: collect_guids(v, found)
    elif isinstance(obj, list):
        for v in obj: collect_guids(v, found)
    return found

def related_guids(element_guid, relationship_type, start_at_end):
    """GUIDs of the elements linked to element_guid by relationship_type (empty when there are none)."""
    result = classification_explorer.get_related_elements(element_guid, relationship_type, start_at_end=start_at_end)
    return collect_guids(result) if isinstance(result, list) else set()

system_guids = {}
unresolved   = []
for row in system_databases:
    guid = guid_of(row['system_qualified_name'])
    if guid:
        system_guids[row['system_qualified_name']] = guid
    else:
        unresolved.append(row)

print(f"Resolved {len(system_guids)} of {len(system_databases)} systems.")
for row in unresolved:
    print(f"   not found: {row['system_qualified_name']}  ({row['estate']} - {row['system_name']})")

----

## Step 1 - A database manager for each system

A software capability describes a piece of software that runs on a server - here the database layer that each system uses to hold its data.  The `DatabaseManager` capability is named after the system it belongs to, and its `resourceName` records the database and schema it manages.

Each step in this workbook looks for what is already there before creating anything, so the workbook can be re-run safely - for example after a new system schema has been added to the CSV file.

----

In [ ]:
# Create the DatabaseManager capabilities and link each one to its system

refresh_tokens()

database_manager_guids = {}
created, linked, present = 0, 0, 0

for row in system_databases:
    system_guid = system_guids.get(row['system_qualified_name'])
    if system_guid is None:
        continue

    capability_qn   = database_manager_qualified_name(row)
    capability_guid = guid_of(capability_qn)
    if capability_guid is None:
        body = {
            "class": "NewElementRequestBody",
            "properties": {
                "class": "DatabaseManagerProperties",
                "qualifiedName": capability_qn,
                "displayName": f"{row['system_name']} Database Manager",
                "description": f"The database layer of {row['system_name']} ({row['estate']}).  It holds the system's data "
                               f"in schema {row['schema_name']} of the {row['database_name']} PostgreSQL database.",
                "deployedImplementationType": DATABASE_MANAGER_TYPE,
                "resourceName": f"{row['database_name']}.{row['schema_name']}"
            }
        }
        capability_guid = asset_maker.create_software_capability(body)
        created += 1
    else:
        present += 1
    database_manager_guids[row['system_qualified_name']] = capability_guid

    # SupportedSoftwareCapability: end 1 is the system, end 2 is the capability
    if system_guid not in related_guids(capability_guid, "SupportedSoftwareCapability", start_at_end=2):
        asset_maker.link_software_capability_to_asset(asset_guid=system_guid, capability_guid=capability_guid)
        linked += 1
        print(f"Linked {row['system_name']} Database Manager to {row['system_name']}  [{row['estate']}]")

print(f"\nDone - {created} database managers created, {present} already present, {linked} SupportedSoftwareCapability relationships created.")

----

## Step 2 - A schema asset for each database manager

Each system's schema is described by a `DeployedDatabaseSchema` asset created from the *PostgreSQL Relational Database Schema* template.  The template's placeholders are replaced with the server, database and schema names, so the new asset gets:

* the qualified name `PostgreSQL Relational Database Schema::Coco PostgreSQL Server 1::<database>.<schema>`;
* a `Connection` to the JDBC resource connector, with an `Endpoint` whose network address is `jdbc:postgresql://host.docker.internal:5442/<database>?currentSchema=<schema>`;
* an embedded connection to the secrets store that supplies the user id and password.

This is the connection the cataloguer uses in the next step - which is why the asset is created from the template rather than built up by hand.

The schema asset is linked to the database manager with `CapabilityAssetUse`, and the use type `OWNS` records that the schema belongs to the system rather than being shared with it.

From here on the workbook runs as Peter Profile, who looks after the data stores and their cataloguing.

----

In [ ]:
# Peter takes over from Gary

connect_as("peterprofile")

In [ ]:
# Create the DeployedDatabaseSchema assets from the template and link them to the database managers

refresh_tokens()

schema_guids = {}
created, linked, present = 0, 0, 0

for row in system_databases:
    capability_guid = database_manager_guids.get(row['system_qualified_name'])
    if capability_guid is None:
        continue

    schema_qn   = schema_qualified_name(row)
    schema_guid = guid_of(schema_qn)
    if schema_guid is None:
        body = {
            "class": "TemplateRequestBody",
            "templateGUID": POSTGRES_SCHEMA_TEMPLATE_GUID,
            "isOwnAnchor": True,
            "placeholderPropertyValues": {
                "hostIdentifier": postgres_host_identifier,
                "portNumber": postgres_port_number,
                "serverName": postgres_server_name,
                "versionIdentifier": "1.0",
                "databaseName": row['database_name'],
                "schemaName": row['schema_name'],
                "schemaDescription": row['schema_description'],
                "secretsStorePathName": secrets_store_path_name,
                "secretsCollectionName": secrets_collection_name
            }
        }
        schema_guid = asset_maker.create_asset_from_template(body)
        created += 1
    else:
        present += 1
    schema_guids[row['system_qualified_name']] = schema_guid

    # CapabilityAssetUse: end 1 is the capability, end 2 is the asset
    if schema_guid not in related_guids(capability_guid, "CapabilityAssetUse", start_at_end=1):
        body = {
            "class": "NewRelationshipRequestBody",
            "properties": {
                "class": "CapabilityAssetUseProperties",
                "useType": "OWNS",
                "description": f"Schema {row['schema_name']} holds the data of {row['system_name']}."
            }
        }
        asset_maker.add_capability_asset_use(software_capability_guid=capability_guid, asset_guid=schema_guid, body=body)
        linked += 1
        print(f"Linked {row['database_name']}.{row['schema_name']} to {row['system_name']} Database Manager  [{row['estate']}]")

print(f"\nDone - {created} schema assets created, {present} already present, {linked} CapabilityAssetUse relationships created.")

----

## Step 3 - Cataloguing the schemas

A `CatalogTarget` relationship tells an integration connector which resources to catalogue.  The `JDBCDatabaseCataloguer` accepts either a whole database or a single schema as a catalog target.  Given a schema, it catalogues only that schema's tables, views and columns, and anchors them to the schema asset - so each system's tables appear under that system's schema and nowhere else.

The catalog target name is the database and schema name, which makes the targets easy to tell apart in the connector's status report below.

----

In [ ]:
# Add each schema asset as a catalog target of the JDBCDatabaseCataloguer

refresh_tokens()

linked, present = 0, 0

for row in system_databases:
    schema_guid = schema_guids.get(row['system_qualified_name'])
    if schema_guid is None:
        continue

    # CatalogTarget: end 1 is the integration connector, end 2 is the element to catalogue
    if JDBC_CATALOGUER_GUID in related_guids(schema_guid, "CatalogTarget", start_at_end=2):
        present += 1
        continue

    body = {
        "class": "NewRelationshipRequestBody",
        "properties": {
            "class": "CatalogTargetProperties",
            "catalogTargetName": f"{row['database_name']}.{row['schema_name']}",
            "permittedSynchronization": "FROM_THIRD_PARTY",
            "deleteMethod": "ARCHIVE"
        }
    }
    asset_maker.add_catalog_target(integration_connector_guid=JDBC_CATALOGUER_GUID, metadata_element_guid=schema_guid, body=body)
    linked += 1
    print(f"Catalog target added: {row['database_name']}.{row['schema_name']}  [{row['estate']}]")

print(f"\nDone - {linked} catalog targets added, {present} already present.")

----

The cataloguer picks up new catalog targets at its next refresh.  The code below asks the integration daemon to refresh it now rather than waiting, which is convenient for this workbook.  There are a lot of tables to catalogue, so the request may time out before the refresh is complete - the connector carries on regardless, and the status report shows its progress.

----

In [ ]:
refresh_tokens()

# Refresh the database integration connector
# An asynchronous call is used since this will return once the refresh is complete - which can result in a timeout on a normal synchronous request.

try:
    print("Refreshing: " + JDBC_CATALOGUER_NAME)
    await runtime_manager._async_refresh_integration_connector(server_guid=integration_daemon_guid, connector_name=JDBC_CATALOGUER_NAME)
except Exception:
    print("Timed out - the connector is still cataloguing.")

print("Completed.")

In [ ]:
# Report on the cataloguer.
# The daemon is looked up by its GUID rather than by name, since the name can match more than one element (see above).

refresh_tokens()

daemon_report = runtime_manager.get_server_report(server_guid=integration_daemon_guid)
connector_reports = daemon_report.get("integrationConnectorReports", []) if isinstance(daemon_report, dict) else []
jdbc_report = next((r for r in connector_reports if r.get("connectorName") == JDBC_CATALOGUER_NAME), None)

def catalog_target_names(obj, found=None):
    """Walk a pyegeria response and gather every 'catalogTargetName' value in it."""
    if found is None: found = set()
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == 'catalogTargetName' and isinstance(v, str): found.add(v)
            else: catalog_target_names(v, found)
    elif isinstance(obj, list):
        for v in obj: catalog_target_names(v, found)
    return found

targets = catalog_target_names(asset_maker.get_catalog_targets(JDBC_CATALOGUER_GUID, graph_query_depth=0))
ours    = {f"{row['database_name']}.{row['schema_name']}" for row in system_databases}

if jdbc_report is None:
    print(f"{JDBC_CATALOGUER_NAME} is not in the report for integration daemon {integration_daemon_guid}")
    print(json.dumps(daemon_report, indent=2)[:2000])
else:
    lines = ["| Connector | Status | Last refresh | Catalog targets | System schemas among them | Exception |",
             "|---|---|---|---|---|---|",
             f"| {JDBC_CATALOGUER_NAME} | {jdbc_report.get('connectorStatus', '---')} | "
             f"{jdbc_report.get('lastRefreshTime', '---')} | {len(targets)} | {len(targets & ours)} of {len(ours)} | "
             f"{jdbc_report.get('failingExceptionMessage') or ''} |"]
    display(Markdown("\n".join(lines)))

----

## Checking the results

The cataloguer names each table after the schema asset it belongs to - `<schema qualified name>::<table name>` - so the tables catalogued for a system can be found by searching for qualified names that start with the schema's qualified name.  The report compares the number found with the number of tables the schema defines.  If some schemas are still short, the connector has not finished: wait a minute and run the cell again.

----

In [ ]:
# Count the tables catalogued under each schema

refresh_tokens()

def catalogued_tables(row):
    """Qualified names of the RelationalTables catalogued under this row's schema asset."""
    result = classification_explorer.find_elements_by_property_value(property_value=schema_qualified_name(row) + "::",
                                                                     property_names=["qualifiedName"],
                                                                     metadata_element_type_name="RelationalTable",
                                                                     starts_with=True,
                                                                     graph_query_depth=0)
    if not isinstance(result, list):
        return []
    return sorted(element.get('properties', {}).get('qualifiedName', '') for element in result)

complete, short = Counter(), []
for row in system_databases:
    if row['system_qualified_name'] not in schema_guids:
        continue
    found    = len(catalogued_tables(row))
    expected = int(row['table_count'])
    if found >= expected:
        complete[row['estate']] += 1
    else:
        short.append((row, found, expected))

lines = ["| Estate | Schemas | Fully catalogued |", "|---|---|---|"]
for estate in ESTATES:
    lines.append(f"| {estate} | {systems_by_estate[estate]} | {complete[estate]} |")
if short:
    lines += ["", "| Schema | System | Tables found | Tables defined |", "|---|---|---|---|"]
    for row, found, expected in short:
        lines.append(f"| `{row['database_name']}.{row['schema_name']}` | {row['system_name']} | {found} | {expected} |")
display(Markdown("\n".join(lines)))

----

## One system's data layer

Here is the result for one system: the Austin LabWare LIMS, which feeds the *Laboratory Test Results* and *Supplier Material Certificates* digital products.  The cell walks from the system to its database manager, from the database manager to its schema, and lists the tables catalogued under the schema with their number of columns.

----

In [ ]:
# Walk from one system to its tables

refresh_tokens()

example = next(row for row in system_databases if row['system_qualified_name'] == "SoftwareServer::AUS-SYS-024::SN-LIM-AU-20190820")

capability_guid = database_manager_guids.get(example['system_qualified_name'])
schema_guid     = schema_guids.get(example['system_qualified_name'])

lines = [f"**{example['system_name']}** (`{example['system_qualified_name']}`)", "",
         f"* Database manager: `{database_manager_qualified_name(example)}` - {capability_guid}",
         f"* Schema asset: `{schema_qualified_name(example)}` - {schema_guid}",
         f"* {example['schema_description']}", "",
         "| Table | Columns |", "|---|---|"]

for table_qn in catalogued_tables(example):
    columns = classification_explorer.find_elements_by_property_value(property_value=table_qn + "::",
                                                                      property_names=["qualifiedName"],
                                                                      metadata_element_type_name="RelationalColumn",
                                                                      starts_with=True,
                                                                      graph_query_depth=0)
    lines.append(f"| {table_qn.rsplit('::', 1)[-1]} | {len(columns) if isinstance(columns, list) else 0} |")

display(Markdown("\n".join(lines)))

----

## In summary

Every system that supplies or receives the strategic digital products now has its data layer described in Egeria: a database manager on the system, the schema it owns, and the tables and columns in that schema, catalogued from the database and kept up to date by the `JDBCDatabaseCataloguer`.

The next step, [strategic-digital-products](../strategic-digital-products/README.md), publishes the data the supply chains carry as digital products.  Each product is filled from the tables catalogued here.

----